# 🗄️ Step 6: Lakebase — Operational Serving

Lakebase is fully managed Postgres integrated with Unity Catalog. In this project it is the
**operational (OLTP) layer** behind the analyst workbench app:

| Direction | What | How |
|---|---|---|
| Lakehouse → Lakebase | Prioritized alert queue (`gold_alert_queue`) + analyst assignments | **Synced tables** (snapshot) |
| App → Lakebase | Analyst triage decisions (`app.analyst_actions`) | Native Postgres `INSERT` |
| Lakebase → Lakehouse | Triage decisions for analytics / Genie | `MERGE` into Delta `analyst_actions` |

**Prerequisite (one-time):** Lakebase Autoscaling project `soc-triage` (branch `production`,
endpoint `primary`) with database `soc_triage`, and the app's service principal added as a Postgres
role. The synced tables are created by this notebook if missing.

In [0]:
%pip install "psycopg[binary]>=3.1" --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
from databricks.sdk import WorkspaceClient
import psycopg

LAKEBASE_BRANCH = "projects/soc-triage/branches/production"
LAKEBASE_ENDPOINT = f"{LAKEBASE_BRANCH}/endpoints/primary"
PG_DATABASE = "soc_triage"
PG_SYNCED_SCHEMA = SCHEMA_NAME          # synced tables land in a Postgres schema named after the UC schema
APP_SERVICE_PRINCIPAL = "da553d37-8a52-4c9e-b8c6-493e4d26addb"  # soc-triage app SP (Postgres role name)

SYNCED_TABLES = {
    # UC synced table name                      : (source Delta table, primary key)
    f"{FULL_SCHEMA}.alert_queue_synced":          (f"{FULL_SCHEMA}.gold_alert_queue", ["event_id"]),
    f"{FULL_SCHEMA}.analyst_assignments_synced":  (f"{FULL_SCHEMA}.analyst_customer_assignments", ["principal", "customer_id"]),
}

w = WorkspaceClient()
me = spark.sql("SELECT current_user()").first()[0]

def pg_connect():
    host = w.api_client.do("GET", f"/api/2.0/postgres/{LAKEBASE_ENDPOINT}")["status"]["hosts"]["host"]
    token = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": LAKEBASE_ENDPOINT})["token"]
    return psycopg.connect(host=host, dbname=PG_DATABASE, user=me, password=token, sslmode="require", autocommit=True)

with pg_connect() as conn:
    print("✅ Connected to Lakebase:", conn.execute("SELECT version()").fetchone()[0][:60])

✅ Connected to Lakebase: PostgreSQL 17.11 (fcae950) on x86_64-pc-linux-gnu, compiled 


In [0]:
# Critical/high alerts (by risk score or ML) with prediction + AI summary — what analysts actually work.
# Written with overwrite (not CREATE OR REPLACE) so the table identity is stable for the synced table.
queue_df = spark.sql(f"""
    SELECT e.event_id, e.event_timestamp, e.event_date, e.customer_id, e.customer_name, e.industry,
           e.source_type, e.risk_tier, e.composite_risk_score, e.threat_intel_match, e.threat_category,
           e.mitre_tactic, e.alert_type, e.action, e.src_ip, e.dst_ip, e.username, e.process_name,
           e.asset_type, e.criticality, p.ml_predicted_severity, p.ml_confidence, s.investigation_summary
    FROM {TABLES['enriched_alerts']} e
    JOIN {TABLES['alert_predictions']} p USING (event_id)
    LEFT JOIN {TABLES['investigation_summaries']} s USING (event_id)
    WHERE e.risk_tier IN ('critical', 'high') OR p.ml_predicted_severity IN ('critical', 'high')
""")
queue_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FULL_SCHEMA}.gold_alert_queue")
spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.gold_alert_queue IS "
          "'Serving table: critical/high alerts with ML prediction + AI summary, synced to Lakebase for the analyst app'")
print(f"✅ gold_alert_queue: {spark.table(f'{FULL_SCHEMA}.gold_alert_queue').count():,} alerts")

✅ gold_alert_queue: 147,317 alerts


In [0]:
import time

def synced_state(name):
    try:
        return w.api_client.do("GET", f"/api/2.0/postgres/synced_tables/{name}")["status"]
    except Exception:
        return None

updates = {}  # synced table -> (pipeline_id, update_id) for refreshes started below
for name, (source, pk) in SYNCED_TABLES.items():
    status = synced_state(name)
    if status is None:
        w.api_client.do("POST", "/api/2.0/postgres/synced_tables", query={"synced_table_id": name}, body={
            "spec": {
                "source_table_full_name": source,
                "primary_key_columns": pk,
                "scheduling_policy": "SNAPSHOT",
                "branch": LAKEBASE_BRANCH,
                "postgres_database": PG_DATABASE,
                "create_database_objects_if_missing": True,
            }
        })
        print(f"🆕 Created synced table {name}")
    else:
        update = w.pipelines.start_update(status["pipeline_id"])  # new snapshot of the source table
        updates[name] = (status["pipeline_id"], update.update_id)
        print(f"🔄 Refresh started for {name} (update {update.update_id})")

# Wait for each refresh's pipeline update to finish (the synced-table status can still show the
# previous snapshot as online), then for newly created tables to come online.
for name in SYNCED_TABLES:
    if name in updates:
        pipeline_id, update_id = updates[name]
        for _ in range(80):
            state = w.pipelines.get_update(pipeline_id, update_id).update.state.value
            if state in ("COMPLETED", "FAILED", "CANCELED"):
                break
            time.sleep(15)
        assert state == "COMPLETED", f"Refresh of {name} ended in state {state}"
    else:
        for _ in range(80):
            state = (synced_state(name) or {}).get("detailed_state", "")
            if state == "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE" or "FAILED" in state:
                break
            time.sleep(15)
        assert "FAILED" not in state, f"Creation of {name} ended in state {state}"
    print(f"   {name}: {state}")

🔄 Refresh started for rajarora_febar.soc_alert_triage.alert_queue_synced (update e2350a2d-41c6-4b54-bee5-8ba0ea40f93a)
🔄 Refresh started for rajarora_febar.soc_alert_triage.analyst_assignments_synced (update b1c39bc5-53d3-42c1-b167-0112f28bd1a3)
   rajarora_febar.soc_alert_triage.alert_queue_synced: COMPLETED
   rajarora_febar.soc_alert_triage.analyst_assignments_synced: COMPLETED


In [0]:
# Synced tables are plain Postgres tables, so UC row filters do NOT apply there. We enforce the same
# customer assignments in Postgres with a view keyed on current_user (the app SP's role). The app SP is
# granted the view only — never the underlying synced table.
APP = f'"{APP_SERVICE_PRINCIPAL}"'
S = PG_SYNCED_SCHEMA

with pg_connect() as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS app")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS app.analyst_actions (
            action_id              TEXT PRIMARY KEY,
            event_id               TEXT NOT NULL,
            analyst_id             TEXT,
            action_type            TEXT CHECK (action_type IN ('escalate','investigate','suppress','false_positive','resolve')),
            severity_override      TEXT,
            notes                  TEXT,
            action_timestamp       TIMESTAMPTZ NOT NULL DEFAULT NOW(),
            time_to_triage_seconds INT
        )""")
    conn.execute("CREATE INDEX IF NOT EXISTS analyst_actions_event_idx ON app.analyst_actions (event_id)")
    conn.execute(f"""
        CREATE OR REPLACE VIEW app.my_alert_queue AS
        SELECT q.* FROM {S}.alert_queue_synced q
        WHERE EXISTS (SELECT 1 FROM {S}.analyst_assignments_synced a
                      WHERE a.principal = current_user AND a.customer_id IN (q.customer_id, '*'))""")
    for stmt in [
        f"GRANT USAGE ON SCHEMA app TO {APP}",
        f"GRANT SELECT ON app.my_alert_queue TO {APP}",
        f"GRANT SELECT, INSERT ON app.analyst_actions TO {APP}",
        f"REVOKE ALL ON ALL TABLES IN SCHEMA {S} FROM {APP}",
    ]:
        conn.execute(stmt)

    print("✅ app.analyst_actions, app.my_alert_queue (RLS view) and app grants in place")
    print("   Rows visible to me (data engineer, '*'):",
          conn.execute("SELECT count(*) FROM app.my_alert_queue").fetchone()[0])
    print("   Customers assigned to the app SP:",
          conn.execute(f"SELECT count(*) FROM {S}.analyst_assignments_synced WHERE principal = %s",
                       (APP_SERVICE_PRINCIPAL,)).fetchone()[0])

✅ app.analyst_actions, app.my_alert_queue (RLS view) and app grants in place
   Rows visible to me (data engineer, '*'): 147317
   Customers assigned to the app SP: 10


In [0]:
from datetime import datetime, timedelta
import random
import uuid

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLES['analyst_actions']} (
    action_id STRING,
    event_id STRING,
    analyst_id STRING,
    action_type STRING COMMENT 'escalate, investigate, suppress, false_positive, resolve',
    severity_override STRING COMMENT 'Analyst can override ML-predicted severity',
    notes STRING,
    action_timestamp STRING,
    time_to_triage_seconds INT COMMENT 'Seconds from alert creation to triage action'
)
COMMENT 'Analyst triage actions — written by the Databricks App to Lakebase, merged back here for analytics'
""")

if spark.table(TABLES["analyst_actions"]).count() == 0:
    random.seed(42)
    sample_event_ids = [r.event_id for r in spark.table(f"{FULL_SCHEMA}.gold_alert_queue").limit(1000).collect()]
    sample_actions = [{
        "action_id": str(uuid.uuid4()),
        "event_id": eid,
        "analyst_id": f"analyst_{random.randint(1, 20)}",
        "action_type": random.choice(["escalate", "investigate", "suppress", "false_positive", "resolve"]),
        "severity_override": random.choice(["critical", "high", "medium", "low", None]),
        "notes": random.choice([
            "Confirmed false positive - known scanner",
            "Escalated to Tier 2 for investigation",
            "Suppressed - customer-approved activity",
            "Resolved - patched vulnerability",
            "Investigating - suspicious lateral movement",
            None,
        ]),
        "action_timestamp": (datetime(2026, 9, 1) + timedelta(
            days=random.randint(0, 29), hours=random.randint(0, 23), minutes=random.randint(0, 59)
        )).strftime("%Y-%m-%dT%H:%M:%S"),
        "time_to_triage_seconds": random.randint(30, 2700),
    } for eid in sample_event_ids]
    (spark.createDataFrame(sample_actions, spark.table(TABLES["analyst_actions"]).schema)
        .write.mode("append").saveAsTable(TABLES["analyst_actions"]))
    print(f"✅ Seeded {len(sample_actions):,} historical analyst actions")
else:
    print(f"ℹ️ analyst_actions already has {spark.table(TABLES['analyst_actions']).count():,} rows — not reseeding")

ℹ️ analyst_actions already has 1,002 rows — not reseeding


In [0]:
import statistics

QUEUE_SQL = """
    SELECT event_id, customer_name, risk_tier, composite_risk_score, ml_predicted_severity, ml_confidence
    FROM app.my_alert_queue
    WHERE risk_tier = 'critical' AND source_type = %s
    ORDER BY composite_risk_score DESC
    LIMIT 50
"""
POINT_SQL = "SELECT investigation_summary FROM app.my_alert_queue WHERE event_id = %s"

with pg_connect() as conn:
    some_event = conn.execute("SELECT event_id FROM app.my_alert_queue LIMIT 1").fetchone()[0]
    conn.execute(QUEUE_SQL, ("edr",)).fetchall()  # warm-up
    timings = {"queue (top 50 critical)": [], "point lookup (by event_id)": []}
    for i in range(50):
        t0 = time.perf_counter(); conn.execute(QUEUE_SQL, (["firewall", "edr", "auth", "email", "cloud_audit"][i % 5],)).fetchall()
        timings["queue (top 50 critical)"].append((time.perf_counter() - t0) * 1000)
        t0 = time.perf_counter(); conn.execute(POINT_SQL, (some_event,)).fetchall()
        timings["point lookup (by event_id)"].append((time.perf_counter() - t0) * 1000)

print("⏱️ Lakebase latency from this notebook (50 runs each, ms):")
for name, ms in timings.items():
    ms.sort()
    print(f"   {name:28s} p50={statistics.median(ms):6.1f}  p95={ms[int(len(ms) * 0.95) - 1]:6.1f}")

⏱️ Lakebase latency from this notebook (50 runs each, ms):
   queue (top 50 critical)      p50=  27.0  p95=  41.1
   point lookup (by event_id)   p50=   2.7  p95=   3.1


In [0]:
with pg_connect() as conn:
    rows = conn.execute("""
        SELECT action_id, event_id, analyst_id, action_type, severity_override, notes,
               to_char(action_timestamp AT TIME ZONE 'UTC', 'YYYY-MM-DD"T"HH24:MI:SS'), time_to_triage_seconds
        FROM app.analyst_actions""").fetchall()

if rows:
    spark.createDataFrame(rows, spark.table(TABLES["analyst_actions"]).schema).createOrReplaceTempView("lakebase_actions")
    spark.sql(f"""
        MERGE INTO {TABLES['analyst_actions']} t USING lakebase_actions s ON t.action_id = s.action_id
        WHEN NOT MATCHED THEN INSERT *
    """)
print(f"✅ Merged {len(rows):,} app triage decisions from Lakebase into {TABLES['analyst_actions']}")

✅ Merged 2 app triage decisions from Lakebase into rajarora_febar.soc_alert_triage.analyst_actions


In [0]:
spark.sql(f"""
    SELECT action_type, COUNT(*) AS count, ROUND(AVG(time_to_triage_seconds) / 60, 1) AS avg_triage_minutes
    FROM {TABLES['analyst_actions']}
    GROUP BY action_type ORDER BY count DESC
""").display()

print("✅ Lakebase serving layer configured!")
print("   - gold_alert_queue synced to Lakebase (snapshot synced table)")
print("   - Customer assignments enforced in Postgres via app.my_alert_queue (current_user)")
print("   - App writes triage decisions to app.analyst_actions; merged back to Delta for analytics")
print("\nNext step: Run 07_dashboard_queries for analytical queries")

action_type,count,avg_triage_minutes
investigate,212,24.6
false_positive,207,21.6
escalate,205,321.0
suppress,197,24.5
resolve,181,22.6


✅ Lakebase serving layer configured!
   - gold_alert_queue synced to Lakebase (snapshot synced table)
   - Customer assignments enforced in Postgres via app.my_alert_queue (current_user)
   - App writes triage decisions to app.analyst_actions; merged back to Delta for analytics

Next step: Run 07_dashboard_queries for analytical queries
